- ## This notebook provides a step-by-step demonstration of the GSE analysis for the SLC Airport. 
- ## By running all the cells of this notebook, you'll be able to get the eGSE charging load profiles, number of required eGSE chargers, and number of required vehicles per GSE type, under the 6 defined charging scenarios.

In [108]:
import geopandas as gpd
import pandas as pd
import datetime
from datetime import time
import get_GSE_tasks as TASK
import get_GSE_events_1_SOC_insufficient as EVENT1
import get_GSE_events_2_charge_during_service_gaps as EVENT2
import get_GSE_events_3_charge_overnight as EVENT3
import math
import warnings
warnings.filterwarnings("ignore")

### Step 1. Scale up the BTS on-time data to include international flights by using the On-Time data.

In [ ]:
# BTS T100 data, which includes total monthly arrival counts (both domestic and international) to each of the airport in 2023, is available at: https://data.nrel.gov/submissions/298 
# Here we use the T100 data for the SLC airport as an example: SLC_T100_data.csv
T100_2023 = pd.read_csv('SLC_T100_data.csv')

# Group month to get the monthly total arrival counts
monthly_arrivals_2023 = T100_2023.groupby(['DEST', 'MONTH']).agg({'DEPARTURES_PERFORMED':'sum'}).reset_index()
monthly_arrivals_2023 = monthly_arrivals_2023.rename(columns={'DEPARTURES_PERFORMED': 'total_arrivals'})

In [52]:
# Domestic arrivals
monthly_arrivals_2023_domestic = T100_2023[T100_2023.ORIGIN_COUNTRY_NAME == 'United States']
monthly_arrivals_2023_domestic = monthly_arrivals_2023_domestic.groupby(['DEST', 'MONTH']).agg({'DEPARTURES_PERFORMED':'sum'}).reset_index()
monthly_arrivals_2023_domestic = monthly_arrivals_2023_domestic.rename(columns={'DEPARTURES_PERFORMED': 'domestic_arrivals'})

In [53]:
# International arrival
monthly_arrivals_2023_international = T100_2023[T100_2023.ORIGIN_COUNTRY_NAME != 'United States']
monthly_arrivals_2023_international = monthly_arrivals_2023_international.groupby(['DEST', 'MONTH']).agg({'DEPARTURES_PERFORMED':'sum'}).reset_index()
monthly_arrivals_2023_international = monthly_arrivals_2023_international.rename(columns={'DEPARTURES_PERFORMED': 'international_arrivals'})

In [54]:
# Merge to get a df that shows the total arrivals, domestic arrivals, and international arrivals
monthly_arrivals_2023 = monthly_arrivals_2023.merge(monthly_arrivals_2023_domestic, on = ['DEST', 'MONTH'], how='outer').merge(monthly_arrivals_2023_international, on=['DEST', 'MONTH'], how='outer').fillna(0)

In [ ]:
# BTS on-time data, which contains the detailed flight arrival info, but only includes domestic flights, is available at: https://data.nrel.gov/submissions/298 
# Here we use the on-time data for the SLC airport as an example:
all_arrivals_ontime_2023 = pd.read_csv('SLC_on_time_flight_data.csv')
monthly_arrivals_ontime_2023 = all_arrivals_ontime_2023.groupby(['Destination Airport', 'MONTH']).size().reset_index(name='Count')
monthly_arrivals_ontime_2023 = monthly_arrivals_ontime_2023.rename(columns={'Destination Airport': 'DEST'})

In [58]:
# Merge the two monthly arrival dfs
monthly_arrivals = monthly_arrivals_ontime_2023.merge(monthly_arrivals_2023, on = ['DEST', 'MONTH'], how = 'left')

# Get the difference between the two monthly arrival dfs
monthly_arrivals['domestic_diff'] = monthly_arrivals['domestic_arrivals'] - monthly_arrivals['Count']
monthly_arrivals['international_diff'] = monthly_arrivals['international_arrivals']

In [60]:
days_in_month = {
    1: 31, 2: 28, 3: 31, 4: 30,
    5: 31, 6: 30, 7: 31, 8: 31,
    9: 30, 10: 31, 11: 30, 12: 31
}

In [61]:
monthly_arrivals['days'] = monthly_arrivals['MONTH'].map(days_in_month)

In [159]:
# get flight arrival distribution
all_arrivals_ontime_2023['Actual Arrival Time'] = all_arrivals_ontime_2023['Actual Arrival Time'].replace('24:00', '00:00')
all_arrivals_ontime_2023['hour'] = all_arrivals_ontime_2023['Actual Arrival Time'].apply(lambda x: x.split(':')[0])
arrival_distribution = all_arrivals_ontime_2023.groupby(['Destination Airport', 'hour']).size().reset_index(name='Count')
total_arrival_per_airport = arrival_distribution.groupby('Destination Airport').agg({'Count': 'sum'}).reset_index()
total_arrival_per_airport = total_arrival_per_airport.rename(columns = {'Count': 'Total_count'})
arrival_distribution = arrival_distribution.merge(total_arrival_per_airport, on = 'Destination Airport', how = 'left')
arrival_distribution['arrival_probability'] = arrival_distribution['Count']/ arrival_distribution['Total_count']
arrival_distribution['hour'] = arrival_distribution['hour'].apply(lambda x: int(x))

In [63]:
# insert flights
import random
from datetime import datetime, timedelta
import numpy as np
from datetime import date

def random_time(df):
    hour = np.random.choice(df['hour'], p=df['arrival_probability'])  # choose hour (0-23) based on the distribution
    minute = random.randint(0, 59)  # Random minute (0-59)
    return f"{hour:02d}:{minute:02d}"  # Format as HH:MM

In [64]:
def insert_arrival(BTS_on_time_df, monthly_arrivals_df, arrival_distribution, airport):
    domes_add_list = []
    inter_add_list = []
    all_flights = pd.DataFrame()
    for i in range(len(monthly_arrivals_df)):
        month = monthly_arrivals_df['MONTH'].iloc[i]
        domestic_diff = monthly_arrivals_df['domestic_diff'].iloc[i]
        international_diff = monthly_arrivals_df['international_diff'].iloc[i]
        days = monthly_arrivals_df['days'].iloc[i]
    
        for i in range(int(domestic_diff)):
            day = random.randint(1, days)
            f_date = date(2023, month, day).strftime("%m/%d/%Y")
            arrival_time = random_time(arrival_distribution)
            domes_add_df = pd.DataFrame({'Carrier Code': ['DDD'], 'Date (MM/DD/YYYY)': [f_date], 'Flight Number': ['0000'],
                                        'Tail Number': ['OOOO'], 'Origin Airport': ['NNN'], 'Actual Arrival Time': [arrival_time],'Destination Airport': [airport]})
            domes_add_list.append(domes_add_df)
        for j in range(int(international_diff)):
            day = random.randint(1, days)
            f_date = date(2023, month, day).strftime("%m/%d/%Y")
            arrival_time = random_time(arrival_distribution)
            inter_add_df = pd.DataFrame({'Carrier Code': ['III'], 'Date (MM/DD/YYYY)': [f_date], 'Flight Number': ['0000'],
                                        'Tail Number': ['OOOO'], 'Origin Airport': ['NNN'], 'Actual Arrival Time': [arrival_time],'Destination Airport': [airport]})
            inter_add_list.append(inter_add_df)
    if len(domes_add_list)>0:
        domes_add = pd.concat(domes_add_list)
        all_flights = pd.concat([BTS_on_time_df, domes_add])
    else:
        all_flights = BTS_on_time_df
        
    if len(inter_add_list)>0:
        inter_add = pd.concat(inter_add_list)
        all_flights = pd.concat([all_flights, inter_add])

    all_flights = all_flights[['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number', 'Origin Airport', 'Actual Arrival Time','Destination Airport']]
    return all_flights

In [65]:
BTS_on_time_dfs = [group for _,group in all_arrivals_ontime_2023.groupby('Destination Airport')]
monthly_arrivals_dfs = [group for _,group in monthly_arrivals.groupby('DEST')]
arrival_distribution_dfs = [group for _,group in arrival_distribution.groupby('Destination Airport')]

In [ ]:
# Scale up the flight arrivals
SLC_all_flights = insert_arrival(all_arrivals_ontime_2023, monthly_arrivals, arrival_distribution, 'SLC')
# SLC_all_flights.to_csv('SLC_all_flight_arrival_data.csv')

### Step 2. Clean up the flight data, remove obvious errors in the flight data
- ##### Fight arrival data for some airports (not all) are problematic, showing unusually high loads at 00:00. This step aims to fix this issue.

In [71]:
def fix_time_format(date_str): # a function to fix "24:00" issue
    if "24:00" in date_str:
        date_part = date_str.split()[0]  # Extract date
        new_date = datetime.strptime(date_part, "%m/%d/%Y") + timedelta(days=1)  # Add a day
        return new_date.strftime("%m/%d/%Y 00:00")  # Format correctly
    return date_str

In [76]:
# Define a function for flight data clean and processing

def data_clean(flight_data_df):
    all_flight_data = flight_data_df
    airport = all_flight_data['Destination Airport'].iloc[0]
    # Fix and parse datetime
    all_flight_data['Arrival_time'] = all_flight_data['Date (MM/DD/YYYY)'] + ' ' + all_flight_data['Actual Arrival Time']
    all_flight_data['Arrival_time'] = all_flight_data['Arrival_time'].apply(fix_time_format)
    all_flight_data['Arrival_time'] = pd.to_datetime(all_flight_data['Arrival_time'])
    
    # Label aircraft type
    all_flight_data['aircraft_type'] = all_flight_data['Carrier Code'].apply(lambda x: 'wide' if x == 'III' else 'narrow')
    
    # Extract hour, day, month
    all_flight_data['Hour'] = all_flight_data['Arrival_time'].dt.hour
    all_flight_data['Day'] = all_flight_data['Arrival_time'].dt.day
    all_flight_data['Month'] = all_flight_data['Arrival_time'].dt.month
    
    # Group by for hourly flight count
    all_flight_data_group = all_flight_data.groupby(['Month', 'Day', 'Hour']).agg({'aircraft_type': 'count'}).reset_index()
    all_flight_data_group = all_flight_data_group.sort_values(by=['Month', 'Day', 'Hour'])
    
    # Define threshold for outlier detection
    hourly_threshold = all_flight_data_group.aircraft_type.quantile(0.9) * 1.35
    
    # Identify outlier day(s)
    outlier_records = all_flight_data_group[(all_flight_data_group['aircraft_type'] > hourly_threshold)&(all_flight_data_group['Hour'] ==0)]
    
    if all_flight_data_group.aircraft_type.quantile(0.9) >= 3 and outlier_records.shape[0] > 0:
        outlier_days = outlier_records[['Month', 'Day']].drop_duplicates()
        
        # Normal days (not outliers)
        normal_days = all_flight_data_group[~all_flight_data_group[['Month', 'Day']].apply(tuple, axis=1).isin(outlier_days.apply(tuple, axis=1))]
        
        # Replace each outlier day with a randomly selected normal day
        for idx, row in outlier_days.iterrows():
            bad_month, bad_day = row.Month, row.Day
        
            # Filter original records for the bad day
            bad_day_data = all_flight_data[(all_flight_data['Month'] == bad_month) & (all_flight_data['Day'] == bad_day)]
        
            # Randomly sample a normal day
            normal_day = normal_days.sample(1).iloc[0]
            norm_month, norm_day = normal_day.Month, normal_day.Day
        
            # Get original normal day data
            normal_day_data = all_flight_data[(all_flight_data['Month'] == norm_month) & (all_flight_data['Day'] == norm_day)].copy()
        
            # Shift normal day datetime to match the outlier date
            delta_days = (pd.Timestamp(f'2023-{bad_month:02d}-{bad_day:02d}') - 
                          pd.Timestamp(f'2023-{norm_month:02d}-{norm_day:02d}')).days
            normal_day_data['Arrival_time'] = normal_day_data['Arrival_time'] + pd.Timedelta(days=delta_days)
        
            # Update date fields
            normal_day_data['Month'] = bad_month
            normal_day_data['Day'] = bad_day
            normal_day_data['Date (MM/DD/YYYY)'] = normal_day_data['Arrival_time'].dt.strftime('%m/%d/%Y')
            normal_day_data['Date (MM/DD/YYYY)'] = normal_day_data['Date (MM/DD/YYYY)'].astype(str)

        
            # Remove bad day and replace it
            all_flight_data = all_flight_data[~((all_flight_data['Month'] == bad_month) & (all_flight_data['Day'] == bad_day))]
            all_flight_data = pd.concat([all_flight_data, normal_day_data], ignore_index=True)
        return all_flight_data
    else:
        return all_flight_data

In [77]:
SLC_all_flights_new = data_clean(SLC_all_flights)

### Step 3. Generate GSE tasks based on flight arrival data

In [82]:
all_flight_data = SLC_all_flights_new.copy()
all_flight_data['Arrival_time'] = all_flight_data['Date (MM/DD/YYYY)'] + ' ' + all_flight_data['Actual Arrival Time']
all_flight_data['Arrival_time'] = all_flight_data['Arrival_time'].apply(fix_time_format)
all_flight_data['Arrival_time'] = pd.to_datetime(all_flight_data['Arrival_time'])
all_flight_data['aircraft_type'] = all_flight_data['Carrier Code'].apply(lambda x: 'wide' if x == 'III' else 'narrow')
SLC_GSE_tasks = TASK.get_gse_schedule(all_flight_data)
   

In [84]:
SLC_GSE_tasks

,GSE_type,airport,airline,tail_number,energy_cons_rate,batt_cap,start_time,end_time,energy_consumption,aircraft_type
0,aircraft tractor,SLC,DL,N369DN,69.00,72.00,2023-01-01 00:45:00,2023-01-01 01:02:00,15.640000,narrow
1,GPU,SLC,DL,N369DN,90.00,160.00,2023-01-01 00:05:00,2023-01-01 00:50:00,50.625000,narrow
2,baggage tractor,SLC,DL,N369DN,22.00,50.00,2023-01-01 00:10:00,2023-01-01 00:48:00,7.663333,narrow
3,baggage tractor,SLC,DL,N369DN,22.00,50.00,2023-01-01 00:10:00,2023-01-01 00:48:00,7.663333,narrow
4,catering truck,SLC,DL,N369DN,100.00,246.67,2023-01-01 00:05:00,2023-01-01 00:36:00,27.383333,narrow
...,...,...,...,...,...,...,...,...,...,...
968419,baggage tractor,SLC,WN,N7715E,22.00,50.00,2023-12-31 23:54:00,2024-01-01 00:32:00,7.663333,narrow
968420,catering truck,SLC,WN,N7715E,100.00,246.67,2023-12-31 23:49:00,2024-01-01 00:20:00,27.383333,narrow
968421,belt loader,SLC,WN,N7715E,1.13,31.70,2023-12-31 23:49:00,2024-01-01 00:46:00,0.750917,narrow
968422,lavatory truck,SLC,WN,N7715E,30.00,40.00,2023-12-31 23:39:00,2023-12-31 23:57:00,2.250000,narrow


### Step 4. Generate GSE events (including both service events and charging events)

In [114]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 1: each GSE vehicle starts charging when its remaining SOC is insufficient for the next service, using 40 kW chargers
def single_airport_add_event(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT1.GSE_assign(df_tmp, 0.1, 0.9, 40, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT1.GSE_assign(df_temp_temp, 0.1, 0.9, 40, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp) 
                    
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [89]:
SLC_GSE_events_S1 = single_airport_add_event(SLC_GSE_tasks)

In [ ]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 2: each GSE vehicle starts charging when its remaining SOC is insufficient for the next service, using 20 kW chargers
def single_airport_add_event_2(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT1.GSE_assign(df_tmp, 0.1, 0.9, 20, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT1.GSE_assign(df_temp_temp, 0.1, 0.9, 20, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp)
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [93]:
SLC_GSE_events_S2 = single_airport_add_event_2(SLC_GSE_tasks)

In [ ]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 3: each GSE vehicle starts charging immediately after it finishes servicing one aircraft, using 40 kW chargers
def single_airport_add_event_3(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT2.GSE_assign_2(df_tmp, 0.1, 0.9, 40, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT2.GSE_assign_2(df_temp_temp, 0.1, 0.9, 40, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp)
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [98]:
SLC_GSE_events_S3 = single_airport_add_event_3(SLC_GSE_tasks)

In [99]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 4: each GSE vehicle starts charging immediately after it finishes servicing one aircraft, using 20 kW chargers
def single_airport_add_event_4(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT2.GSE_assign_2(df_tmp, 0.1, 0.9, 20, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT2.GSE_assign_2(df_temp_temp, 0.1, 0.9, 20, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp)
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [100]:
SLC_GSE_events_S4 = single_airport_add_event_4(SLC_GSE_tasks)

In [101]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 5: GSE vehicles charge between 10 pm to 8 am, using 40 kW chargers

def single_airport_add_event_5(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT3.GSE_assign_3(df_tmp, 0.1, 0.9, 40, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT3.GSE_assign_3(df_temp_temp, 0.1, 0.9, 40, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp)
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [102]:
SLC_GSE_events_S5 = single_airport_add_event_5(SLC_GSE_tasks)

In [103]:
# Define a function to generate GSE events based on GSE_tasks under charging scenario 5: GSE vehicles charge between 10 pm to 8 am, using 20 kW chargers

def single_airport_add_event_6(GSE_tasks):
    tasks_per_airline = [group for _,group in GSE_tasks.groupby('airline')] # task df of each airline
    GSE_events_list = []
    airport = GSE_tasks['airport'].iloc[0]
    for j in tasks_per_airline:
        list_of_gse = [group for _,group in j.groupby('GSE_type')]
        airline = j['airline'].iloc[0]
        for df_tmp in list_of_gse:
            if df_tmp['GSE_type'].iloc[0] in ['baggage tractor', 'catering truck']:
                df_tmp['task_idx'] = ["service_" + str(m) for m in range(df_tmp.shape[0])]
                df_assign_result_tmp, veh_num_init = EVENT3.GSE_assign_3(df_tmp, 0.1, 0.9, 20, 1)
                df_assign_result_tmp['airline'] = airline
                GSE_events_list.append(df_assign_result_tmp)
            else:
                task_per_gse_per_aircraft_type = [group for _,group in df_tmp.groupby('aircraft_type')]
                for df_temp_temp in task_per_gse_per_aircraft_type:
                    df_temp_temp['task_idx'] = ["service_" + str(m) for m in range(df_temp_temp.shape[0])]
                    df_assign_result_tmp, veh_num_init = EVENT3.GSE_assign_3(df_temp_temp, 0.1, 0.9, 20, 1)
                    df_assign_result_tmp['airline'] = airline
                    GSE_events_list.append(df_assign_result_tmp) 
    GSE_events = pd.concat(GSE_events_list)
    return GSE_events

In [104]:
SLC_GSE_events_S6 = single_airport_add_event_6(SLC_GSE_tasks)

### Step 5. Postprocess the charging events for the overnight charging scenarios S5 and S6

In [110]:
# Postprocess the charging events for S5
charge_power = 40
all_charge_events = SLC_GSE_events_S5[SLC_GSE_events_S5.Task_type == 'Charge']
all_charge_events['date'] = all_charge_events['start_time'].dt.date
charge_events_per_day = [group for _,group in all_charge_events.groupby('date')]

processed_charge_events_list = []
for i in range(len(charge_events_per_day)):
    df_temp = charge_events_per_day[i]
    date_part  = df_temp['start_time'].iloc[0].date()
    
    chg_events_10 = df_temp[(df_temp['start_time'].dt.hour == 22) & (df_temp['start_time'].dt.minute == 0)]
    if len(chg_events_10)>0:
        chg_events_10['energy'] = (chg_events_10['end_time'] - chg_events_10['start_time'] + timedelta(minutes=1)).dt.total_seconds()/3600*charge_power
        total_energy = chg_events_10['energy'].sum()
        avg_power = total_energy/10/0.94
        total_charger_num = math.ceil(avg_power/charge_power)
        time_part = time(22, 0, 0)
        charge_start_time = datetime.datetime.combine(date_part, time_part)
        charge_duration = total_energy/(total_charger_num*charge_power)
        charge_end_time = charge_start_time + pd.Timedelta(hours=charge_duration)
        charge_end_time = charge_end_time
        charge_events_10 = pd.DataFrame({'Task_type':['Charge']*total_charger_num,'Idx':['charge_all_10pm']*total_charger_num,'SOC_start':[0]*total_charger_num,'SOC_end':[0.9]*total_charger_num,'start_time':[charge_start_time]*total_charger_num,'end_time':[charge_end_time]*total_charger_num})

        chg_events_other = df_temp[~((df_temp['start_time'].dt.hour == 22) & (df_temp['start_time'].dt.minute == 0))]
        chg_events_all = pd.concat([chg_events_other, charge_events_10])
    else:
        chg_events_all = df_temp
    processed_charge_events_list.append(chg_events_all)

SLC_processed_charge_events_S5 = pd.concat(processed_charge_events_list)

In [112]:
charge_power = 20

all_charge_events = SLC_GSE_events_S6[SLC_GSE_events_S6.Task_type == 'Charge']
all_charge_events['date'] = all_charge_events['start_time'].dt.date
charge_events_per_day = [group for _,group in all_charge_events.groupby('date')]

processed_charge_events_list = []
for i in range(len(charge_events_per_day)):
    df_temp = charge_events_per_day[i]
    date_part  = df_temp['start_time'].iloc[0].date()
    
    chg_events_10 = df_temp[(df_temp['start_time'].dt.hour == 22) & (df_temp['start_time'].dt.minute == 0)]
    if len(chg_events_10)>0:
        chg_events_10['energy'] = (chg_events_10['end_time'] - chg_events_10['start_time'] + timedelta(minutes=1)).dt.total_seconds()/3600*charge_power
        total_energy = chg_events_10['energy'].sum()
        avg_power = total_energy/10/0.94
        total_charger_num = math.ceil(avg_power/charge_power)
        time_part = time(22, 0, 0)
        charge_start_time = datetime.datetime.combine(date_part, time_part)
        charge_duration = total_energy/(total_charger_num*charge_power)
        charge_end_time = charge_start_time + pd.Timedelta(hours=charge_duration)
        charge_end_time = charge_end_time
        charge_events_10 = pd.DataFrame({'Task_type':['Charge']*total_charger_num,'Idx':['charge_all_10pm']*total_charger_num,'SOC_start':[0]*total_charger_num,'SOC_end':[0.9]*total_charger_num,'start_time':[charge_start_time]*total_charger_num,'end_time':[charge_end_time]*total_charger_num})

        chg_events_other = df_temp[~((df_temp['start_time'].dt.hour == 22) & (df_temp['start_time'].dt.minute == 0))]
        chg_events_all = pd.concat([chg_events_other, charge_events_10])
    else:
        chg_events_all = df_temp
    processed_charge_events_list.append(chg_events_all)

SLC_processed_charge_events_S6 = pd.concat(processed_charge_events_list)

### Step 6. Generate the GSE load profiles based on GSE charging events under the 6 charging scenarios.

In [113]:
# Define a function to generate GSE load profiles
def load_profile(charge_events, charger_power):
    
    # Round to minute precision
    charge_events['start_time'] = charge_events['start_time'].dt.floor('min')
    charge_events['end_time'] = charge_events['end_time'].dt.floor('min')

    # Mark +1 at start and -1 at end
    event_changes = pd.concat([
        charge_events['start_time'].value_counts().sort_index().rename("change"),
        charge_events['end_time'].value_counts().sort_index().rename("change") * -1
    ]).groupby(level=0).sum()
    
    # Compute cumulative sum to get concurrent events
    active_events = event_changes.cumsum()
    
    # Fill in missing minutes for continuity
    active_events = active_events.asfreq('T', method='pad').fillna(0)
    full_range = pd.date_range(start="2023-01-01 00:00:00", end="2023-12-31 23:59:00", freq='T')
    active_events = active_events.reindex(full_range, fill_value=0)

    
    active_df = active_events.rename("num_active_events").reset_index()
    active_df.columns = ["time_stamp", "count"]
    active_df['power'] = active_df['count']*charger_power
    
    return active_df

In [115]:
SLC_GSE_events_S1

,Task_type,Idx,SOC_start,SOC_end,start_time,end_time,veh_id,GSE_type,aircraft_type,airline
0,Service,service_0,0.900000,0.583594,2023-01-01 00:17:00,2023-01-01 01:02:00,1,GPU,narrow,AA
1,Service,service_1,0.583594,0.267188,2023-01-01 12:38:00,2023-01-01 13:23:00,1,GPU,narrow,AA
2,Charge,charge_0,0.267188,0.900000,2023-01-01 13:28:00,2023-01-01 16:09:00,1,GPU,narrow,AA
3,Service,service_5,0.900000,0.583594,2023-01-01 16:49:00,2023-01-01 17:34:00,1,GPU,narrow,AA
4,Service,service_6,0.583594,0.267188,2023-01-01 21:39:00,2023-01-01 22:24:00,1,GPU,narrow,AA
...,...,...,...,...,...,...,...,...,...,...
246,Service,service_237,0.750000,0.712500,2023-12-31 10:27:00,2023-12-31 10:42:00,1,water truck,narrow,YV
247,Charge,charge_11,0.712500,0.900000,2023-12-31 10:47:00,2023-12-31 10:58:00,1,water truck,narrow,YV
248,Service,service_21,0.900000,0.862500,2023-02-11 12:38:00,2023-02-11 12:53:00,2,water truck,narrow,YV
249,Service,service_53,0.862500,0.825000,2023-02-28 14:43:00,2023-02-28 14:58:00,2,water truck,narrow,YV


In [116]:
# GSE load profile for charging scenario 1
SLC_load_profile_S1 = load_profile(SLC_GSE_events_S1[SLC_GSE_events_S1['Task_type']=='Charge'], 40)

In [119]:
# GSE load profile for charging scenario 2
SLC_load_profile_S2 = load_profile(SLC_GSE_events_S2[SLC_GSE_events_S2['Task_type']=='Charge'], 20)

In [126]:
# GSE load profile for charging scenario 3
SLC_load_profile_S3 = load_profile(SLC_GSE_events_S3[SLC_GSE_events_S3['Task_type']=='Charge'], 40)

In [127]:
# GSE load profile for charging scenario 4
SLC_load_profile_S4 = load_profile(SLC_GSE_events_S4[SLC_GSE_events_S4['Task_type']=='Charge'], 20)

In [138]:
# GSE load profile for charging scenario 5
SLC_load_profile_S5 = load_profile(SLC_processed_charge_events_S5[SLC_processed_charge_events_S5['Task_type']=='Charge'], 40)

In [139]:
# GSE load profile for charging scenario 6
SLC_load_profile_S6 = load_profile(SLC_processed_charge_events_S6[SLC_processed_charge_events_S6['Task_type']=='Charge'], 20)

### Step 7. Estimate the number of chargers needed under the 6 charging scenarios

In [145]:
SLC_GSE_charger_count_S1 = SLC_load_profile_S1['count'].max()
SLC_GSE_charger_count_S2 = SLC_load_profile_S2['count'].max()
SLC_GSE_charger_count_S3 = SLC_load_profile_S3['count'].max()
SLC_GSE_charger_count_S4 = SLC_load_profile_S4['count'].max()
SLC_GSE_charger_count_S5 = SLC_load_profile_S5['count'].max()
SLC_GSE_charger_count_S6 = SLC_load_profile_S6['count'].max()

In [148]:
SLC_GSE_charger_count_S3

np.int64(147)

### Step 8. Estimate the number of vehicles needed per GSE type under the 6 charging scenarios

In [154]:
# Define a function to calculate the number of vehicles needed

def cal_veh_count(GSE_events):
 
    GPU_counts_narrow = 0
    GPU_counts_wide = 0
    aircraft_tractor_counts_narrow = 0
    aircraft_tractor_counts_wide = 0
    baggage_tractor_counts = 0
    cargo_loader_counts_wide = 0
    belt_loader_counts_narrow = 0
    belt_loader_counts_wide = 0
    cater_truck_counts = 0
    lav_truck_counts_narrow = 0
    lav_truck_counts_wide = 0
    water_truck_counts_narrow = 0
    water_truck_counts_wide = 0

    # GSE vehicles are shared within each airline
    airline_groups = [group for _,group in GSE_events.groupby('airline')]
    # wide-body and narrow-body aircraft use the same type of baggage tractors and catering trucks
    for i in range(len(airline_groups)):
        GSE_events_i = airline_groups[i]
        baggage_tractor_counts += GSE_events_i[GSE_events_i.GSE_type == 'baggage tractor'].veh_id.max()
        cater_truck_counts += GSE_events_i[GSE_events_i.GSE_type == 'catering truck'].veh_id.max()
    
    # wide-body and narrow-body aircraft use different types of other GSE vehicles
    narrow_body_events = GSE_events[GSE_events['aircraft_type'] == 'narrow']
    wide_body_events = GSE_events[GSE_events['aircraft_type'] == 'wide']
    narrow_body_events_airline_groups = [group for _,group in narrow_body_events.groupby('airline')]
    wide_body_events_airline_groups = [group for _,group in wide_body_events.groupby('airline')]

    for j in range(len(narrow_body_events_airline_groups)):
        GSE_events_narrow_j = narrow_body_events_airline_groups[j]
        GPU_counts_narrow += GSE_events_narrow_j[GSE_events_narrow_j.GSE_type == 'GPU'].veh_id.max()
        aircraft_tractor_counts_narrow += GSE_events_narrow_j[GSE_events_narrow_j.GSE_type == 'aircraft tractor'].veh_id.max()
        belt_loader_counts_narrow += GSE_events_narrow_j[GSE_events_narrow_j.GSE_type == 'belt loader'].veh_id.max()
        lav_truck_counts_narrow += GSE_events_narrow_j[GSE_events_narrow_j.GSE_type == 'lavatory truck'].veh_id.max()
        water_truck_counts_narrow += GSE_events_narrow_j[GSE_events_narrow_j.GSE_type == 'water truck'].veh_id.max()
    for k in range(len(wide_body_events_airline_groups)):
        GSE_events_wide_k = wide_body_events_airline_groups[k]
        GPU_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'GPU'].veh_id.max()
        aircraft_tractor_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'aircraft tractor'].veh_id.max()
        belt_loader_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'belt loader'].veh_id.max()
        lav_truck_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'lavatory truck'].veh_id.max()
        water_truck_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'water truck'].veh_id.max()
        cargo_loader_counts_wide += GSE_events_wide_k[GSE_events_wide_k.GSE_type == 'cargo loader'].veh_id.max()
        
    GSE_vehicle_count_list = {'baggage_tractor_counts': int(baggage_tractor_counts), 'cater_truck_counts': int(cater_truck_counts), \
                              'GPU_counts_narrow': int(GPU_counts_narrow), 'aircraft_tractor_counts_narrow': int(aircraft_tractor_counts_narrow), \
                              'belt_loader_counts_narrow': int(belt_loader_counts_narrow), 'lav_truck_counts_narrow': int(lav_truck_counts_narrow), \
                              'water_truck_counts_narrow': int(water_truck_counts_narrow), \
                              'GPU_counts_wide': int(GPU_counts_wide), 'aircraft_tractor_counts_wide': int(aircraft_tractor_counts_wide), \
                              'belt_loader_counts_wide': int(belt_loader_counts_wide), 'lav_truck_counts_wide': int(lav_truck_counts_wide), \
                              'water_truck_counts_wide': int(water_truck_counts_wide), 'cargo_loader_counts_wide': int(cargo_loader_counts_wide)
                               }
    GSE_vehicle_count_df = pd.DataFrame([GSE_vehicle_count_list])
    return GSE_vehicle_count_df

In [157]:
SLC_GSE_vehicle_count_S1 = cal_veh_count(SLC_GSE_events_S1)
SLC_GSE_vehicle_count_S2 = cal_veh_count(SLC_GSE_events_S2)
SLC_GSE_vehicle_count_S3 = cal_veh_count(SLC_GSE_events_S3)
SLC_GSE_vehicle_count_S4 = cal_veh_count(SLC_GSE_events_S4)
SLC_GSE_vehicle_count_S5 = cal_veh_count(SLC_GSE_events_S5)
SLC_GSE_vehicle_count_S6 = cal_veh_count(SLC_GSE_events_S6)